# HW2: Word2Vector
## Packages

In [ ]:
import json
import string
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from collections import Counter
import jieba
import nltk
from scipy.stats import cosine
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Pre-produce
###  1.1 Create Vocabulary Table

In [ ]:
# load txt text from ./data 
def load_data(path, version='zh'):
    with open(path+version+'.txt', 'r', encoding='utf-8') as f:
        text = f.read()
    return text

# split text to words
def preprocess_text(text, language='zh'):
    if language == 'zh':
        words = jieba.cut(text)
        return list(words)
    else:
        # split
        words = nltk.word_tokenize(text)
        # remove stopwords and punctuations
        stop_words = set(nltk.corpus.stopwords.words('english'))
        return [word.strip(string.punctuation) 
                for word in words 
                if word.strip(string.punctuation) not in stop_words and word.strip(string.punctuation) not in string.whitespace]

# build vocab
def build_vocab(words):
    # count word frequency
    counter = Counter(words)
    # vocab is a dict{word1:num1, word2:num2, ...}, and freq in desc order  
    vocab = {word:i+1 for i, word in enumerate(counter.most_common())}
    return vocab

### 1.2 Create Training Data

In [ ]:
# create context-center word pairs([context_nums], center_num)
def create_training_data(words, vocab, window_size=5):
    data = []
    for center_idx in range(window_size, len(words)-window_size):
        # get context words and center word
        context = words[center_idx+1:center_idx+window_size+1] + words[center_idx-window_size:center_idx]
        center = words[center_idx]
        # get context words and center word's nums
        context_nums = [vocab.get(word) for word in context]
        center_nums = vocab.get(center)
        # return pairs
        data.append((context_nums, center_nums))
        return data

##  2. CBOW Model
### Use CBOW model to do this task.

In [ ]:
# CBOW model
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        super(CBOW, self).__init__()
        self.embedding_size = embedding_size
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.linear = nn.Linear(embedding_size, vocab_size)
        
    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings_mean = embeddings.mean(dim=1, keepdim=True)
        out = self.linear(embeddings_mean)
        log_probs = torch.log_softmax(out, dim=1)
        return log_probs

## 3. Optimizer & Loss Function


In [ ]:
# optimizer & loss function
def get_optim_and_loss():
    # optimizer
    optimizer = torch.optim.SGD(lr=0.001, momentum=0.9)
    # loss
    criterion = nn.CrossEntropyLoss()
    return optimizer, criterion

##  4. Train

In [ ]:
# train
def train(model, data, optimizer, criterion, epochs, lossv, batch_size=64):
    total_loss = 0.0
    for epoch in range(epochs):
        for i in tqdm(range(0, len(data), batch_size)):
            # a batch of pairs
            batch = data[i:i+batch_size]
            # turn context into tensor
            contexts = torch.tensor([context for context,_ in batch], dtype=torch.long).to(device)
            centers = torch.tensor([center for _, center in batch], dtype=torch.long).to(device)
            optimizer.zero_grad()
            log_probs = model(contexts)
            loss = criterion(log_probs, centers)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        lossv.append(total_loss/len(data))
        print(f'epoch {epoch}, loss: {total_loss/len(data)}')
    print('Training finished')

## 5. Save Word Vectors

In [ ]:
# save word vectors
def save_word_vectors(model, vocab, save_vec_path, version="zh"):
    word_vectors = {}
    # get embbing weights
    embbeding_weights = model.embeddings.weight.data
    for word, i in vocab.items():
        word_vectors[word] = embbeding_weights[i].numpy()
    
    # save as json format
    with open(save_vec_path+version, 'w', encoding='utf-8') as f:
        json.dump(word_vectors, f, ensure_ascii=False, indent=4)

# load word vectors
def load_word_vectors(save_vec_path):
    with open(save_vec_path, 'r', encoding='utf-8') as f:
        word_vectors = json.load(f)
    return word_vectors
        
# save model
def save_model(model, save_model_path, version="zh"):
    torch.save(model.state_dict(), save_model_path+version)

# load model
def load_model(model, model_path, version="zh"):
    return model.load_state_dict(torch.load(model_path+version))

## 6. Start Word2vec Train From Here

In [ ]:
# start word2vec train process from here
txt_path = "data/"
save_vec_path = "model/word2vec/"
save_model_path = "model/"

### Select version to execute ('zh' or 'en')

### zh

In [ ]:
# load text
zh_text = load_data(txt_path, version='zh')

In [ ]:
# create words
zh_words = preprocess_text(zh_text, language='zh')

In [ ]:
# create vocab
zh_vocab = build_vocab(zh_words)

In [ ]:
# create training data
zh_train_data = create_training_data(zh_words, zh_vocab, window_size=5)

In [ ]:
# optimizer & loss function
optimizer, criterion = get_optim_and_loss()

In [ ]:
# train zh model
model_zh = CBOW(vocab_size=len(zh_vocab), embedding_size=50).to(device)
epochs = 10
# set train mode
model_zh.train()
# start train
lossv_zh = []
print('Start zh training...')
train(model_zh, zh_train_data, optimizer, criterion, epochs, lossv_zh, batch_size=64)

In [ ]:
# plot 
plt.plot([epoch+1 for epoch in epochs], lossv_zh)
plt.title('CBOW Training Loss (en)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
# save
save_model(model_zh, save_model_path, version='zh')
save_word_vectors(model_zh, zh_vocab, save_vec_path, version='zh')

### en

In [ ]:
# load text
en_text = load_data(txt_path, version='en')

In [ ]:
# create words
en_words = preprocess_text(en_text, language='en')

In [ ]:
# create vocab
en_vocab = build_vocab(en_words)

In [ ]:
# create training data
en_train_data = create_training_data(en_words, en_vocab, window_size=5)

In [ ]:
# optimizer & loss function
optimizer, criterion = get_optim_and_loss()

In [ ]:
# train en model
model_en = CBOW(vocab_size=len(en_vocab), embedding_size=50).to(device)
epochs = 10
# set train mode
model_en.train()
# start train
lossv_en = []
print('Start en training...')
train(model_en, zh_train_data, optimizer, criterion, epochs, lossv_en, batch_size=64)

In [ ]:
# plot 
plt.plot([epoch + 1 for epoch in epochs], lossv_en)
plt.title('CBOW Training Loss (en)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
# save
save_model(model_en, save_model_path, version='en')
save_word_vectors(model_en, en_vocab, save_vec_path, version='en')

## 7. An Example Task to Use Trained Word Vectors
### 7.1 Compute similarity among words

In [ ]:
similarities = {}
# find similar words
def find_similar_words(word, word_vectors, top_n=5):
    word_vec = word_vectors.get(word)
    for target_word, target_vec in word_vec.items():
        if target_word != word:
            similarity = cosine(target_vec, word_vec)
            similarities[target_word] = similarity
            
    top_similarities = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return top_similarities

### 7.2 Demo


In [ ]:
# load word vectors
word_vectors_en = load_word_vectors(save_vec_path+"en")
word_vectors_zh = load_word_vectors(save_vec_path+"zh")
# find similar top n words
en_word = "people"
zh_word = "人民"
top_n = 5

# english
top_similarities = find_similar_words(en_word, word_vectors_en, top_n)
print(f"Most similar top {top_n} words to '{en_word}':")
for word, similarity in top_similarities:
    print(f"{word}: {similarity:.3f}")
    
# chinese
top_similarities = find_similar_words(zh_word, word_vectors_zh, top_n)
print(f"Most similar top {top_n} words to '{zh_word}':")
for word, similarity in top_similarities:
    print(f"{word}: {similarity:.3f}")